# 12 - Equilibria And Incubation Fitting Quickstart

## Workflow

1. Load the copied example files, reference them to Fc/Fc+, and select the Ar and CO2 traces.
2. Build fit-ready CV inputs from segment 1 only, trimmed from -1.0 to -1.6 V vs Fc/Fc+.
3. Fit the Ar trace first to get the Co/CoRed potential and tied diffusion estimate.
4. Define balanced reversible chemistry with reaction-local `K` and `k_exchange` values.
5. Fit the CO2 series with shared chemical parameters and per-CV CO2/Cdl values, then inspect entered and equilibrated concentrations.

The dissolved CO2 values below are tutorial placeholders. Replace `CO2_DISSOLVED_MOL_M3` with Henry-law or calibrated dissolved concentrations for the real experiment.

## Imports And Paths

In [1]:
from copy import deepcopy
from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd

import ecat as e

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "examples" / "data" / "co_zn_co2_cv"
EXPORT_DIR = ROOT / "notebooks" / "_outputs"
EXPORT_DIR.mkdir(exist_ok=True)

HAS_ELECTROKITTY = importlib.util.find_spec("electrokitty") is not None

e.plotting_style("notebook")

print("eCAT version:", getattr(e, "__version__", "unknown"))
print("Example data:", DATA_DIR.relative_to(ROOT))
print("ElectroKitty available:", HAS_ELECTROKITTY)

eCAT version: 0.1.0b6
Example data: examples/data/co_zn_co2_cv
ElectroKitty available: True


## Load And Reference The CVs

These files are CHI text exports copied into `examples/data/co_zn_co2_cv`. The reference correction uses the Fc wave and stores the active axis as potential vs Fc/Fc+.

In [2]:
cvs = e.get_data({
    "folder path": str(DATA_DIR),
    "reference mode": "keyword",
    "reference keyword": "Fc",
    "reference guess": 0.4,
    "reference label": "Fc/Fc+",
    "electrode diameter": 0.3,
    "print": False,
})
cvs = e.filter(cvs, {"segments": 3}, {"print": False})

CO2_LABELS = ["5 % CO2", "10 % CO2", "20 % CO2", "40 % CO2", "70 % CO2", "100 % CO2"]
CO2_DISSOLVED_MOL_M3 = [14.0, 28.0, 56.0, 112.0, 196.0, 280.0]

ar_reference_cv = e.filter(cvs, {"gas": "Ar"}, {"print": False})[0]
co2_group = e.filter(cvs, {"compounds": "CO2"}, {"print": False})
co2_group = e.sort(co2_group, ["concentrations"], {"print": False})
selected_cvs = [ar_reference_cv, *co2_group]
selected_labels = ["Ar", *CO2_LABELS]

summary_rows = []
for label, cv in zip(selected_labels, selected_cvs):
    x = np.asarray(cv.x(), dtype=float)
    summary_rows.append({
        "Label": label,
        "Gas": getattr(cv, "gas", ""),
        "Scan Rate / V s^-1": getattr(cv, "scan_rate", np.nan),
        "Potential Min vs Fc / V": np.nanmin(x),
        "Potential Max vs Fc / V": np.nanmax(x),
        "Compounds": "; ".join(getattr(cv, "compounds", []) or []),
        "Concentrations": "; ".join(getattr(cv, "concentrations", []) or []),
    })

display(pd.DataFrame(summary_rows))

,Label,Gas,Scan Rate / V s^-1,Potential Min vs Fc / V,Potential Max vs Fc / V,Compounds,Concentrations
0,Ar,Ar,0.1,-1.7380,0.4620,TBAPF6; Fc; Co(dmgH)2(py)Cl,0.1 M; 3 mM; 1 mM
1,5 % CO2,Ar/CO2,0.1,-1.7170,0.4830,TBAPF6; Fc; Co(dmgH)2(py)Cl; Zn(cyclen)(OTf)2;...,0.1 M; 3 mM; 1 mM; 1 mM; 2.8 M; 5 %
2,10 % CO2,Ar/CO2,0.1,-1.7175,0.4825,TBAPF6; Fc; Co(dmgH)2(py)Cl; Zn(cyclen)(OTf)2;...,0.1 M; 3 mM; 1 mM; 1 mM; 2.8 M; 10 %
3,20 % CO2,Ar/CO2,0.1,-1.7180,0.4820,TBAPF6; Fc; Co(dmgH)2(py)Cl; Zn(cyclen)(OTf)2;...,0.1 M; 3 mM; 1 mM; 1 mM; 2.8 M; 20 %
4,40 % CO2,Ar/CO2,0.1,-1.7185,0.4815,TBAPF6; Fc; Co(dmgH)2(py)Cl; Zn(cyclen)(OTf)2;...,0.1 M; 3 mM; 1 mM; 1 mM; 2.8 M; 40 %
5,70 % CO2,Ar/CO2,0.1,-1.7185,0.4815,TBAPF6; Fc; Co(dmgH)2(py)Cl; Zn(cyclen)(OTf)2;...,0.1 M; 3 mM; 1 mM; 1 mM; 2.8 M; 70 %
6,100 % CO2,Ar/CO2,0.1,-1.7195,0.4805,TBAPF6; Fc; Co(dmgH)2(py)Cl; Zn(cyclen)(OTf)2;...,0.1 M; 3 mM; 1 mM; 1 mM; 2.8 M; 100 %


## Trim The Fit Window Before Fitting

The fitting inputs use eCAT's 1-based segment numbering. Segment 1 is the initial sweep from positive potential toward the lower vertex. `cv_data()` applies the internal eCAT trim from -1.0 to -1.6 V vs Fc/Fc+ before fitting.

In [3]:
FIT_SEGMENT = 1
FIT_START_VS_FC = -1.0
FIT_LOWER_VS_FC = -1.6
AR_FIT_STRIDE = 20
CO2_FIT_STRIDE = 25
CHEMICAL_INCUBATION_TIME_S = 0.0

FIT_WINDOW_VS_FC = [FIT_START_VS_FC, FIT_LOWER_VS_FC]
REAL_CV_WINDOW_OPTIONS = {
    "segment": FIT_SEGMENT,
    "potential window": FIT_WINDOW_VS_FC,
    "trim mode": "strict",
}

print(f"Fit segment: {FIT_SEGMENT}")
print(f"Requested fit window vs Fc/Fc+: {FIT_WINDOW_VS_FC[0]:.3f} to {FIT_WINDOW_VS_FC[1]:.3f} V")
print(f"Homogeneous chemical incubation before quiet time: {CHEMICAL_INCUBATION_TIME_S:g} s")

ax = e.multiplot(selected_cvs, {
    "labels": selected_labels,
    "plot segment": FIT_SEGMENT,
    "title": "Referenced CVs: Segment 1 Fit Region",
    "print": False,
})
ax.set_xlim(FIT_WINDOW_VS_FC[0], FIT_WINDOW_VS_FC[1])

Fit segment: 1
Requested fit window vs Fc/Fc+: -1.000 to -1.600 V
Homogeneous chemical incubation before quiet time: 0 s


(-1.0, -1.6)

## Build Fit-Ready Inputs

`cv_data()` performs the actual trim before the fitting algorithm sees the data. Cdl is estimated as a total capacitance in `F` from the full referenced CV first, then copied onto the trimmed fit input. eCAT keeps `cell.Cdl` in total farads, with electrode area separately in `cell.A` (`m^2`), and converts to backend area-normalized capacitance internally when needed. The fit input uses the same FOWA background option name, `background correction: "start current"`, to subtract the first selected current point after trimming and before stride. We also attach calibrated dissolved-CO2 values in `mol/m^3` to each CO2 input so `fit_cvs()` can map them to `concentrations.bulk.CO2`.



In [4]:
def fit_ready_input(cv, stride, label, co2_concentration=None):
    cdl_source = e.simulation.cv_data(cv, {
        "stride": stride,
        "estimate Cdl": "auto",
    })
    input_obj = e.simulation.cv_data(cv, {
        **REAL_CV_WINDOW_OPTIONS,
        "stride": stride,
        "estimate Cdl": False,
        "background correction": "start current",
        "incubation time": CHEMICAL_INCUBATION_TIME_S,
    })
    for key in ("estimated_cdl", "estimated_Cdl", "estimated_cdl_diagnostics", "estimated_cdl_error"):
        if key in cdl_source.metadata:
            input_obj.metadata[key] = cdl_source.metadata[key]
    if co2_concentration is not None:
        input_obj.metadata.setdefault("concentrations", {})["CO2"] = float(co2_concentration)
    input_obj.metadata["short_label"] = label
    return input_obj


ar_input = fit_ready_input(ar_reference_cv, AR_FIT_STRIDE, "Ar")
co2_inputs = [
    fit_ready_input(cv, CO2_FIT_STRIDE, label, co2_concentration)
    for cv, label, co2_concentration in zip(co2_group, CO2_LABELS, CO2_DISSOLVED_MOL_M3)
]

input_rows = []
for input_obj in [ar_input, *co2_inputs]:
    meta = input_obj.metadata
    input_rows.append({
        "Label": meta.get("short_label"),
        "Source": meta.get("name"),
        "Requested Window / V": meta.get("potential_window_requested"),
        "Effective Window / V": meta.get("potential_window_effective"),
        "Segment": meta.get("segments"),
        "Stride": meta.get("stride"),
        "Points": len(input_obj.E),
        "Background": meta.get("background_correction"),
        "Estimated Cdl / F": meta.get("estimated_cdl"),
        "CO2 / mol m^-3": meta.get("concentrations", {}).get("CO2", np.nan),
    })

display(pd.DataFrame(input_rows))

,Label,Source,Requested Window / V,Effective Window / V,Segment,Stride,Points,Background,Estimated Cdl / F,CO2 / mol m^-3
0,Ar,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_-1...,"[-1.0, -1.6]","[-1.5999999999999999, -1.0]",1,20,31,start current,0.000028,NaN
1,5 % CO2,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1m...,"[-1.0, -1.6]","[-1.6, -1.0]",1,25,25,start current,0.000039,14.0
2,10 % CO2,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1m...,"[-1.0, -1.6]","[-1.5995, -1.0005]",1,25,25,start current,0.000028,28.0
3,20 % CO2,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1m...,"[-1.0, -1.6]","[-1.6, -1.0]",1,25,25,start current,0.000037,56.0
4,40 % CO2,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1m...,"[-1.0, -1.6]","[-1.5995, -1.0005]",1,25,25,start current,0.000038,112.0
5,70 % CO2,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1m...,"[-1.0, -1.6]","[-1.5995, -1.0005]",1,25,25,start current,0.000031,196.0
6,100 % CO2,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1m...,"[-1.0, -1.6]","[-1.5995, -1.0005]",1,25,25,start current,0.000026,280.0


## Fit 1: Ar Co/CoRed Redox Baseline

The Ar trace has the catalyst redox couple without the Zn/CO2 chemistry. Fit the Co/CoRed formal potential and one tied diffusion value first. The cell block is written out explicitly here so the physical defaults are visible.

In [5]:
ar_redox_params = {
    "concentrations": {"bulk": {"Co": 1.0, "CoRed": 0.0}},
    "diffusion": {"Co": 1.5e-9, "CoRed": 1.5e-9},
    "kinetics": [
        {"E0": -1.5, "k0": 1e-3, "alpha": 0.5},
    ],
    "cell": {"T": "auto", "Ru": "auto", "Cdl": "auto", "A": "auto"},
    "spatial": "fast",
}

single_ar_fit = None
if not HAS_ELECTROKITTY:
    print('Install ecat-electrochemistry[simulation] to run the simulation fitting cells.')
else:
    single_ar_fit = e.simulation.fit_cv(
        ar_input,
        "E",
        ar_redox_params,
        fit={
            "vary": ["E0_0", "D"],
            "fixed": {"alpha_*": 0.5, "k0_*": 1e-3},
            "bounds": "auto",
            "transform": "auto",
        },
        options={
            "plot": True,
            "post correction": "offset",
            "max nfev": 20,
        },
    )

,Parameter,Control,Value
0,Fit strategy,method,least squares
1,Simulation backend,backend,electrokitty
2,Mechanism,mechanism,E(1):Co=CoRed
3,Mechanism preset,compile_mechanism(...).preset,E
4,Residual mode,"options[""residual""]",direct
5,Final current correction,"options[""post correction""]",offset
6,Residual normalization,"options[""residual normalization""]",max abs measured
7,Evaluation budget,"options[""max nfev""]",~60
8,Data points,input.E,31
9,Fit targets,"fit[""vary""]",2


,Path,Parameter,Initial,Lower,Upper,Transform
0,kinetics.0.E0,E⁰,-1.5 V,-1.6 V,-1 V,linear
1,"diffusion.Co, diffusion.CoRed",D (tied),1.5e-09 m²/s,1.5e-11 m²/s,1.5e-07 m²/s,log10


,Path,Parameter,Value
0,kinetics.0.alpha,α,0.5
1,kinetics.0.k0,k⁰,0.001 m/s


,Group,Path,Parameter,Fit Status,Initial Value,Final Value
0,spatial,spatial.dx_fraction,Δx/xmax,fixed,0.005,
1,spatial,spatial.nx,nₓ,fixed,8,
2,spatial,spatial.viscosity,η,fixed,4.7e-07 m²/s,
3,spatial,spatial.rotation,ω,fixed,0 Hz,


,Group,Path,Parameter,Fit Status,Initial Value,Final Value
0,cell,cell.T,T,fixed,298 K,
1,cell,cell.Ru,Rᵤ,fixed,0 Ω,
2,cell,cell.Cdl,Cdl,fixed,2.7823e-05 F,
3,cell,cell.A,A,fixed,7.06858e-06 m²,


,Group,Path,Parameter,Fit Status,Initial Value,Final Value
0,bulk,diffusion.Co,D(Co),fit-tied,1.5e-09 m²/s,1.23422e-09 m²/s
1,bulk,diffusion.CoRed,D(CoRed),fit-tied,1.5e-09 m²/s,1.23422e-09 m²/s
2,bulk,concentrations.bulk.Co,[Co],fixed,1 mol/m³,
3,bulk,concentrations.bulk.CoRed,[CoRed],fixed,0 mol/m³,


,Group,Path,Step,Parameter,Fit Status,Initial Value,Final Value
0,kinetics,kinetics.0.E0,0,E⁰,fit,-1.5 V,-1.49216 V
1,kinetics,kinetics.0.k0,0,k⁰,fixed,0.001 m/s,
2,kinetics,kinetics.0.alpha,0,α,fixed,0.5,


## Define The Co/Zn/H2O/CO2 Mechanism

The model starts from the Ar redox fit. Reversible chemical steps use dimensionless activity `K` plus the standard-state exchange frequency `k_exchange`; eCAT solves the selected reaction equations before the CV and compiles each entry to backend `kf`/`kb` rates. Every participating species must have an entered concentration, including explicit zeros.

The acid/base bookkeeping is written explicitly: `ZnOH2 + H2O = ZnOH + H3O`, `ZnOH + CO2 = ZnHCO3`, `ZnHCO3 + H2O = ZnOH2 + HCO3`, and `HCO3 + H3O = CO2 + 2H2O`. The last reaction closes a dependent thermodynamic cycle. It is marked `equilibrate=False`, so it remains reversible during incubation and the CV but does not add an independent initial-speciation constraint while another `K` in that cycle is being fitted.

`incubation_time` belongs to each `SimulatedCVInput`. When it is nonzero, eCAT first solves the selected pre-equilibria and then evolves bulk homogeneous chemical steps for that time before the backend quiet-time hold. Surface and mixed-phase steps remain backend-only.

In [6]:
def average_diffusion(params, default=1.5e-9):
    values = []
    for name, value in (params.get("diffusion", {}) or {}).items():
        if str(name).startswith("Co") and np.isfinite(float(value)):
            values.append(float(value))
    return float(np.mean(values)) if values else default


ar_best_params = deepcopy(single_ar_fit.best_params if single_ar_fit is not None else ar_redox_params)
D_CO = average_diffusion(ar_best_params)
kinetics_from_ar = deepcopy(ar_best_params.get("kinetics", ar_redox_params["kinetics"]))

co_zn_co2_mechanism = "\n".join([
    "E(1):Co=CoRed",
    "C:CoRed+ZnOH2>CoH+ZnOH",
    "C:CoH+ZnOH2>H2+Co+ZnOH",
    "C:ZnOH2+H2O=ZnOH+H3O",
    "C:ZnOH+CO2=ZnHCO3",
    "C:ZnHCO3+H2O=ZnOH2+HCO3",
    "C:HCO3+H3O=CO2+2H2O",
    "C:Zn+H2O=ZnOH2",
])

ZN_DEPROTONATION_K = 1e-8
ZN_CO2_K = 1e2
ZNHCO3_HYDRATION_K = 1.0
CARBON_ACID_K = 1.0 / (ZN_DEPROTONATION_K * ZN_CO2_K * ZNHCO3_HYDRATION_K)
ZN_HYDRATION_K = 1e2

zn_group_params = {
    "concentrations": {
        "bulk": {
            "Co": 1.0,
            "CoRed": 0.0,
            "CoH": 0.0,
            "H2": 0.0,
            "Zn": 1.0,
            "ZnOH2": 0.0,
            "ZnOH": 0.0,
            "ZnHCO3": 0.0,
            "H2O": 2800.0,
            "CO2": CO2_DISSOLVED_MOL_M3[-1],
            "H3O": 0.0,
            "HCO3": 0.0,
        },
    },
    "diffusion": {
        "Co": D_CO,
        "CoRed": D_CO,
        "CoH": D_CO,
        "H2": 4.5e-9,
        "Zn": 1.0e-9,
        "ZnOH2": 1.0e-9,
        "ZnOH": 1.0e-9,
        "ZnHCO3": 1.0e-9,
        "H2O": 2.0e-9,
        "CO2": 2.0e-9,
        "H3O": 9.0e-9,
        "HCO3": 1.2e-9,
    },
    "kinetics": kinetics_from_ar,
    "reactions": {
        "CoRed+ZnOH2>CoH+ZnOH": {"k": 1.0},
        "CoH+ZnOH2>H2+Co+ZnOH": {"k": 1.0},
        "ZnOH2+H2O=ZnOH+H3O": {"K": ZN_DEPROTONATION_K, "k_exchange": 100.0},
        "ZnOH+CO2=ZnHCO3": {"K": ZN_CO2_K, "k_exchange": 3000.0},
        "ZnHCO3+H2O=ZnOH2+HCO3": {"K": ZNHCO3_HYDRATION_K, "k_exchange": 1000.0},
        "HCO3+H3O=CO2+2H2O": {"K": CARBON_ACID_K, "k_exchange": 100.0, "equilibrate": False},
        "Zn+H2O=ZnOH2": {"K": ZN_HYDRATION_K, "k_exchange": 1.0},
    },
    "activity": {
        "standard concentration": 1000.0,
        "gamma": {},
    },
    "cell": "auto",
    "spatial": "fast",
}

chemical_steps = [line.split(":", 1)[1] for line in co_zn_co2_mechanism.splitlines() if line.startswith("C:")]
reaction_rows = []
for index, (reaction, entry) in enumerate(zn_group_params["reactions"].items()):
    reaction_rows.append({
        "Step": index,
        "Reaction": reaction,
        "Initial Equilibrium": entry.get("equilibrate", True) if "K" in entry else "not reversible",
        "k": entry.get("k"),
        "K": entry.get("K"),
        "k_exchange / s^-1": entry.get("k_exchange"),
    })

display(pd.DataFrame(reaction_rows))

,Step,Reaction,Initial Equilibrium,k,K,k_exchange / s^-1
0,0,CoRed+ZnOH2>CoH+ZnOH,not reversible,1.0,NaN,NaN
1,1,CoH+ZnOH2>H2+Co+ZnOH,not reversible,1.0,NaN,NaN
2,2,ZnOH2+H2O=ZnOH+H3O,True,NaN,1.000000e-08,100.0
3,3,ZnOH+CO2=ZnHCO3,True,NaN,1.000000e+02,3000.0
4,4,ZnHCO3+H2O=ZnOH2+HCO3,True,NaN,1.000000e+00,1000.0
5,5,HCO3+H3O=CO2+2H2O,False,NaN,1.000000e+06,100.0
6,6,Zn+H2O=ZnOH2,True,NaN,1.000000e+02,1.0


## Fit 2: CO2 Series With Shared Equilibria

The `fit` spec decides what is optimized. `per_cv` only marks which paths are dataset-specific. Here, each CV gets its own dissolved CO2 concentration and automatically estimated cell capacitance. The redox kinetics, entered Zn total, other equilibrium constants, hydride-formation rate, and H2-formation rate remain shared unless listed in `vary`.

The cycle-closing acid/base reaction is dynamic-only during fitting. If every cycle `K` is known independently, they must satisfy the thermodynamic cycle relation before all are used as simultaneous pre-equilibrium constraints.

In [7]:
co2_group_fit = None
if not HAS_ELECTROKITTY:
    print('Install ecat-electrochemistry[simulation] to run the simulation fitting cells.')
else:
    co2_group_fit = e.simulation.fit_cvs(
        co2_inputs,
        co_zn_co2_mechanism,
        zn_group_params,
        fit={
            "vary": [
                "reactions.0.k",
                "reactions.3.K",
                "reactions.4.K",
            ],
            "fixed": {"alpha_*": 0.5, "k0_*": 1e-3},
            "bounds": "auto",
            "transform": "auto",
        },
        per_cv=["cell.Cdl", "cell.Ru", "concentrations.bulk.CO2"],
        options={
            "plot": True,
            "post correction": "offset",
            "max nfev": 20,
            "print stats": False,
            "print corrections": False,
            "print progress": False,
        },
    )
    co2_group_fit.show({
        "print setup": False,
        "print stats": True,
        "print corrections": True,
        "print params": True,
        "print simulation": False,
    })

,CV,Label,Source,Input Concentrations,Mapped Concentrations,Points
0,1,5 % CO2,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1mMZn(cyclen)(OTf)2_2.8MH2O_5%CO2_-1.3_to_0.9V_100mVs,0.1 M TBAPF6; 3 mM Fc; 1 mM Co(dmgH)2(py)Cl; 1 mM Zn(cyclen)(OTf)2; 2.8 M H2O; 5 % CO2,H2O → H2O: 2800 mol/m³; CO2 → CO2: 14 mol/m³,25
1,2,10 % CO2,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1mMZn(cyclen)(OTf)2_2.8MH2O_10%CO2_-1.3_to_0.9V_100mVs,0.1 M TBAPF6; 3 mM Fc; 1 mM Co(dmgH)2(py)Cl; 1 mM Zn(cyclen)(OTf)2; 2.8 M H2O; 10 % CO2,H2O → H2O: 2800 mol/m³; CO2 → CO2: 28 mol/m³,25
2,3,20 % CO2,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1mMZn(cyclen)(OTf)2_2.8MH2O_20%CO2_-1.3_to_0.9V_100mVs,0.1 M TBAPF6; 3 mM Fc; 1 mM Co(dmgH)2(py)Cl; 1 mM Zn(cyclen)(OTf)2; 2.8 M H2O; 20 % CO2,H2O → H2O: 2800 mol/m³; CO2 → CO2: 56 mol/m³,25
3,4,40 % CO2,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1mMZn(cyclen)(OTf)2_2.8MH2O_40%CO2_-1.3_to_0.9V_100mVs,0.1 M TBAPF6; 3 mM Fc; 1 mM Co(dmgH)2(py)Cl; 1 mM Zn(cyclen)(OTf)2; 2.8 M H2O; 40 % CO2,H2O → H2O: 2800 mol/m³; CO2 → CO2: 112 mol/m³,25
4,5,70 % CO2,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1mMZn(cyclen)(OTf)2_2.8MH2O_70%CO2_-1.3_to_0.9V_100mVs,0.1 M TBAPF6; 3 mM Fc; 1 mM Co(dmgH)2(py)Cl; 1 mM Zn(cyclen)(OTf)2; 2.8 M H2O; 70 % CO2,H2O → H2O: 2800 mol/m³; CO2 → CO2: 196 mol/m³,25
5,6,100 % CO2,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1mMZn(cyclen)(OTf)2_2.8MH2O_100%CO2_-1.3_to_0.9V_100mVs,0.1 M TBAPF6; 3 mM Fc; 1 mM Co(dmgH)2(py)Cl; 1 mM Zn(cyclen)(OTf)2; 2.8 M H2O; 100 % CO2,H2O → H2O: 2800 mol/m³; CO2 → CO2: 280 mol/m³,25


,Group,Path,Parameter,Fit Status,Initial Value,Final Value
0,spatial,spatial.dx_fraction,Δx/xmax,fixed,0.005,
1,spatial,spatial.nx,nₓ,fixed,8,
2,spatial,spatial.viscosity,η,fixed,4.7e-07 m²/s,
3,spatial,spatial.rotation,ω,fixed,0 Hz,


,Group,Path,Parameter,Fit Status,Initial Value,Final Value
0,cell,cell.Cdl,Cdl,fixed,3.9045e-05 F,
1,cell,cell.T,T,fixed,298 K,
2,cell,cell.A,A,fixed,7.06858e-06 m²,
3,cell,cell.Ru,Rᵤ,fixed,0 Ω,


,Group,Path,Parameter,Fit Status,Initial Value,Final Value
0,bulk,diffusion.Co,D(Co),fixed,1.23422e-09 m²/s,
1,bulk,diffusion.CoRed,D(CoRed),fixed,1.23422e-09 m²/s,
2,bulk,diffusion.CoH,D(CoH),fixed,1.23422e-09 m²/s,
3,bulk,diffusion.H2,D(H2),fixed,4.5e-09 m²/s,
4,bulk,diffusion.Zn,D(Zn),fixed,1e-09 m²/s,
5,bulk,diffusion.ZnOH2,D(ZnOH2),fixed,1e-09 m²/s,
6,bulk,diffusion.ZnOH,D(ZnOH),fixed,1e-09 m²/s,
7,bulk,diffusion.ZnHCO3,D(ZnHCO3),fixed,1e-09 m²/s,
8,bulk,diffusion.H2O,D(H2O),fixed,2e-09 m²/s,
9,bulk,diffusion.CO2,D(CO2),fixed,2e-09 m²/s,


,Group,Path,Step,Parameter,Fit Status,Initial Value,Final Value
0,kinetics,kinetics.0.E0,0,E⁰,fixed,-1.49216 V,
1,kinetics,kinetics.0.k0,0,k⁰,fixed,0.001 m/s,
2,kinetics,kinetics.0.alpha,0,α,fixed,0.5,
3,reactions,reactions.0.k,0,k₁,fit,1 s⁻¹,1762.2 s⁻¹
4,reactions,reactions.1.k,1,k₂,fixed,1 s⁻¹,
5,reactions,reactions.2.K,2,k₃,fixed,1e-08 s⁻¹,
6,reactions,reactions.2.k_exchange,2,k_exchange,fixed,100 s⁻¹,
7,reactions,reactions.3.K,3,k₄,fit,100 s⁻¹,61.6623 s⁻¹
8,reactions,reactions.3.k_exchange,3,k_exchange,fixed,3000 s⁻¹,
9,reactions,reactions.4.K,4,k₅,fit,1 s⁻¹,0.389505 s⁻¹


...

,Group,Path,Parameter,Fit Status,Initial Value,Final Value
0,spatial,spatial.dx_fraction,Δx/xmax,fixed,0.005,
1,spatial,spatial.nx,nₓ,fixed,8,
2,spatial,spatial.viscosity,η,fixed,4.7e-07 m²/s,
3,spatial,spatial.rotation,ω,fixed,0 Hz,


,Group,Path,Parameter,Fit Status,Initial Value,Final Value
0,cell,cell.Cdl,Cdl,fixed,3.9045e-05 F,
1,cell,cell.T,T,fixed,298 K,
2,cell,cell.A,A,fixed,7.06858e-06 m²,
3,cell,cell.Ru,Rᵤ,fixed,0 Ω,


,Group,Path,Parameter,Fit Status,Initial Value,Final Value
0,bulk,diffusion.Co,D(Co),fixed,1.23422e-09 m²/s,
1,bulk,diffusion.CoRed,D(CoRed),fixed,1.23422e-09 m²/s,
2,bulk,diffusion.CoH,D(CoH),fixed,1.23422e-09 m²/s,
3,bulk,diffusion.H2,D(H2),fixed,4.5e-09 m²/s,
4,bulk,diffusion.Zn,D(Zn),fixed,1e-09 m²/s,
5,bulk,diffusion.ZnOH2,D(ZnOH2),fixed,1e-09 m²/s,
6,bulk,diffusion.ZnOH,D(ZnOH),fixed,1e-09 m²/s,
7,bulk,diffusion.ZnHCO3,D(ZnHCO3),fixed,1e-09 m²/s,
8,bulk,diffusion.H2O,D(H2O),fixed,2e-09 m²/s,
9,bulk,diffusion.CO2,D(CO2),fixed,2e-09 m²/s,


,Group,Path,Step,Parameter,Fit Status,Initial Value,Final Value
0,kinetics,kinetics.0.E0,0,E⁰,fixed,-1.49216 V,
1,kinetics,kinetics.0.k0,0,k⁰,fixed,0.001 m/s,
2,kinetics,kinetics.0.alpha,0,α,fixed,0.5,
3,reactions,reactions.0.k,0,k₁,fit,1 s⁻¹,1762.2 s⁻¹
4,reactions,reactions.1.k,1,k₂,fixed,1 s⁻¹,
5,reactions,reactions.2.K,2,k₃,fixed,1e-08 s⁻¹,
6,reactions,reactions.2.k_exchange,2,k_exchange,fixed,100 s⁻¹,
7,reactions,reactions.3.K,3,k₄,fit,100 s⁻¹,61.6623 s⁻¹
8,reactions,reactions.3.k_exchange,3,k_exchange,fixed,3000 s⁻¹,
9,reactions,reactions.4.K,4,k₅,fit,1 s⁻¹,0.389505 s⁻¹


,CV,Label,Path,Parameter,Fit Status,Initial Value,Final Value,Source
0,1,5 % CO2,cell.Cdl,Cdl,fixed,3.9045e-05 F,,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1mMZn(cyclen)(OTf)2_2.8MH2O_5%CO2_-1.3_to_0.9V_100mVs
1,1,5 % CO2,cell.Ru,Rᵤ,fixed,0 Ω,,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1mMZn(cyclen)(OTf)2_2.8MH2O_5%CO2_-1.3_to_0.9V_100mVs
2,1,5 % CO2,concentrations.bulk.CO2,[CO2],fixed,14 mol/m³,,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1mMZn(cyclen)(OTf)2_2.8MH2O_5%CO2_-1.3_to_0.9V_100mVs
3,2,10 % CO2,cell.Cdl,Cdl,fixed,2.7743e-05 F,,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1mMZn(cyclen)(OTf)2_2.8MH2O_10%CO2_-1.3_to_0.9V_100mVs
4,2,10 % CO2,cell.Ru,Rᵤ,fixed,0 Ω,,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1mMZn(cyclen)(OTf)2_2.8MH2O_10%CO2_-1.3_to_0.9V_100mVs
5,2,10 % CO2,concentrations.bulk.CO2,[CO2],fixed,28 mol/m³,,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1mMZn(cyclen)(OTf)2_2.8MH2O_10%CO2_-1.3_to_0.9V_100mVs
6,3,20 % CO2,cell.Cdl,Cdl,fixed,3.711e-05 F,,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1mMZn(cyclen)(OTf)2_2.8MH2O_20%CO2_-1.3_to_0.9V_100mVs
7,3,20 % CO2,cell.Ru,Rᵤ,fixed,0 Ω,,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1mMZn(cyclen)(OTf)2_2.8MH2O_20%CO2_-1.3_to_0.9V_100mVs
8,3,20 % CO2,concentrations.bulk.CO2,[CO2],fixed,56 mol/m³,,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1mMZn(cyclen)(OTf)2_2.8MH2O_20%CO2_-1.3_to_0.9V_100mVs
9,4,40 % CO2,cell.Cdl,Cdl,fixed,3.829e-05 F,,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMCo(dmgH)2(py)Cl_1mMZn(cyclen)(OTf)2_2.8MH2O_40%CO2_-1.3_to_0.9V_100mVs


## Inspect Per-CV Concentrations

This final table checks both input mapping and conservation. `best_params_by_cv` contains the entered per-CV CO2 values; each final `simulation_result.params` contains the equilibrated concentrations passed to the backend. The summed Zn-bearing and carbon-bearing concentrations replace the old explicit pool-total bookkeeping.

In [8]:
if co2_group_fit is None:
    print("Run the CO2 group-fit cell to inspect fitted per-CV parameters.")
else:
    rows = []
    for dataset, entered_i, result_i in zip(
        co2_group_fit.datasets,
        co2_group_fit.best_params_by_cv,
        co2_group_fit.simulation_results,
    ):
        entered_bulk = entered_i["concentrations"]["bulk"]
        equilibrated_bulk = result_i.params["concentrations"]["bulk"]
        rows.append({
            "Label": dataset.get("label", ""),
            "Entered CO2 / mol m^-3": entered_bulk["CO2"],
            "Equilibrated CO2 / mol m^-3": equilibrated_bulk["CO2"],
            "Zn-bearing total / mol m^-3": sum(
                equilibrated_bulk[name] for name in ["Zn", "ZnOH2", "ZnOH", "ZnHCO3"]
            ),
            "Carbon-bearing total / mol m^-3": sum(
                equilibrated_bulk[name] for name in ["CO2", "ZnHCO3", "HCO3"]
            ),
            "Estimated Cdl / F": entered_i["cell"].get("Cdl"),
        })
    display(pd.DataFrame(rows))
    co2_group_fit.simulation_results[-1].show({
        "print setup": False,
        "print params": False,
        "print states": True,
    })

,Label,Entered CO2 / mol m^-3,Equilibrated CO2 / mol m^-3,Zn-bearing total / mol m^-3,Carbon-bearing total / mol m^-3,Estimated Cdl / F
0,5 % CO2,14.0,13.838663,1.0,14.0,0.000039
1,10 % CO2,28.0,27.771398,1.0,28.0,0.000028
2,20 % CO2,56.0,55.676298,1.0,56.0,0.000037
3,40 % CO2,112.0,111.541841,1.0,112.0,0.000038
4,70 % CO2,196.0,195.393656,1.0,196.0,0.000031
5,100 % CO2,280.0,279.275150,1.0,280.0,0.000026


,Phase,Species,Entered,Equilibrated,Incubated
0,bulk,Zn,1 mol/m³,0.00355932 mol/m³,0.00355932 mol/m³
1,bulk,ZnOH2,0 mol/m³,0.995741 mol/m³,0.995741 mol/m³
2,bulk,ZnOH,0 mol/m³,3.84285e-05 mol/m³,3.84285e-05 mol/m³
3,bulk,ZnHCO3,0 mol/m³,0.000661768 mol/m³,0.000661768 mol/m³
4,bulk,H2O,2800 mol/m³,2797.55 mol/m³,2797.55 mol/m³
5,bulk,CO2,280 mol/m³,279.275 mol/m³,279.275 mol/m³
6,bulk,H3O,0 mol/m³,0.724888 mol/m³,0.724888 mol/m³
7,bulk,HCO3,0 mol/m³,0.724188 mol/m³,0.724188 mol/m³
